# NB6c · Multi-turn evals from scratch: a simulated user
**Lecture 6 · Observability and Evals · Agentic Systems Engineering · Dr. Rajat Dandekar, Vizuara**

Friday's incident needs two turns to test: Emma asks, sees a preview, then says yes. Real conversations are longer
and messier: she changes her mind, answers a question with a question, pushes to skip a step. Agents fail **between**
turns. This notebook builds a multi-turn eval step by step:

1. **scripted turns**: fixed messages; simple, and where they stop working
2. **a simulated user**: a model plays Emma, with a goal, things she knows, and a way of writing
3. **scoring the whole conversation**: the end state, the order of actions across turns, what the agent asked
4. the same conversation under **Thursday's prompt**, and our 20 recorded conversations
5. the same idea in **DeepEval** (conversational test cases, its simulator), and where it shows up in **Langfuse**

**Two ways to run it.** With your Argus key everything is live (through Argus's gateway). With no key it replays our
recorded run.

In [1]:
# Setup: install the Ledgerline sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Ledgerline sandbox + reference agent)
try:
    import ledgerline  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ledgerline-erp[all] @ git+{COURSE_REPO}.git#subdirectory=ledgerline"], check=True)
    import ledgerline
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("LEDGER_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Ledgerline", ledgerline.__version__, "· model mode:", os.environ["LEDGER_LLM_MODE"])

Ledgerline 0.1.0 · model mode: auto


**Install DeepEval** (pinned) and choose live or replay: in Argus press **Notebook key** and paste its two lines below.

In [2]:
import subprocess, sys, os, json
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deepeval==4.3.4", "anthropic==2.8.0"], check=True)
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

ARGUS_URL = ""     # e.g. "https://rajatworkspace--northbridge-argus.us-east.modal.run"
ARGUS_KEY = ""     # your Argus key, "ar-..."; leave empty to replay our recorded run

from ledgerline.course.l6.nbkit import connect, agent_turns, text_of, read_verdict
kit = connect(ARGUS_URL, ARGUS_KEY)
SMALL = "claude-haiku-5-5"      # plays Emma and judges; the agent itself stays on claude-sonnet-5-5

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


Mode: LIVE through https://rajatworkspace--northbridge-argus.us-east.modal.run/gateway


## 1 · Scripted turns, and where they stop working
The simplest multi-turn test is a script: message one, message two. It works when we know what the agent will say.
Here Emma gives no date. Watch what the agent does with her first message, and how the script's second message fits.

In [3]:
script = ["Order the usual, please.", "Yes, go ahead."]
scripted = kit.run("nb6c_scripted.json", lambda: agent_turns(kit, "good", script, tracing=False))
for t in scripted["turns"]:
    print("Emma (script):", t["text"])
    print("Ask Northbridge:", t["reply"][:300].replace("\n", " "), "...\n")
print("Saved:", scripted["saved"])

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:  20%|██        | 1/5 [00:00<00:00,  5.57it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:02<00:00,  2.12it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:02<00:00,  2.20it/s]

Emma (script): Order the usual, please.
Ask Northbridge: - The usual order is 2,400 x BAT-500 battery packs, booked to PRD-03 (£680,000 budget left), with Voltway Cells as supplier. - I don't have a need-by date for it. What date should I use? - BAT-500 isn't below its reorder level. Free stock is 6,600 and 800 more are on order.  Once I have the date, I' ...

Emma (script): Yes, go ahead.
Ask Northbridge: - I still need a need-by date. "Yes" didn't give me one, and I won't guess it. - Everything else is ready:   - 2,400 x BAT-500 to PRD-03, with £680,000 budget left.   - Supplier is Voltway Cells (VC-EIN-0042), and its UN 38.3 certificate is valid. - Give me the date and I'll preview it with the gros ...

Saved: []


If the agent asked a question in turn 1 (in our runs it often asks for the need-by date), the script still answered
"Yes, go ahead." A script cannot listen. To test a conversation we need a user who **reads the reply and reacts**.

## 2 · A simulated user
We let a model play Emma. It gets three things, and nothing else:

- a **goal**: what she wants out of this chat
- what she **knows**: the facts she can answer questions with
- a **persona**: how she writes

It is told to write one short message at a time and to say `DONE` when her goal is met. Below are one goal and one
persona from the course's set (`ledgerline/course/l6/convo.py` has five goals and three personas).

In [4]:
from ledgerline.course.l6.convo import SCENARIOS, PERSONAS, SIM
goal = SCENARIOS["change_mind"]
print("GOAL   :", goal["goal"])
print("KNOWS  :", goal["knows"])
print("PERSONA:", PERSONAS["impatient"])
print("\nThe instructions the simulated Emma gets:\n")
print(SIM.format(goal=goal["goal"], knows=goal["knows"], persona=PERSONAS["impatient"]))

GOAL   : Order your usual battery packs, needed by 4 Dec 2026, but only half the usual quantity this time.
KNOWS  : Start by just asking for the usual. Only when you SEE the preview with 2,400 do you remember you need 1,200 this time: ask to change it to 1,200. Confirm only a preview that shows 1,200 BAT-500 on PRD-03.
PERSONA: In a hurry between meetings. You push to skip steps ('just do it', 'no need to show me'), but you still answer a direct question when you must.

The instructions the simulated Emma gets:

You are playing Emma Clarke, a purchase executive at Northbridge Cycles in Manchester. You are typing to your work
assistant, Ask Northbridge, in a chat window. Stay in character as Emma for the whole conversation.

Your goal today: Order your usual battery packs, needed by 4 Dec 2026, but only half the usual quantity this time.
What you know: Start by just asking for the usual. Only when you SEE the preview with 2,400 do you remember you need 1,200 this time: ask to change it 

**Emma's side, in code.** One function: given the conversation so far, ask the model for Emma's next message. The roles
are swapped on purpose: from the simulator's point of view, *Emma's* messages are its own (role `assistant`) and the
agent's replies are what it reads (role `user`).

In [5]:
def emma_says(history, goal, persona):
    import anthropic
    messages = [{"role": "user", "content": "[The chat window is open. Write your first message to the assistant.]"}]
    for h in history:
        messages.append({"role": "assistant" if h["who"] == "emma" else "user", "content": h["text"]})
    reply = anthropic.Anthropic().messages.create(model=SMALL, max_tokens=1500, messages=messages,
                system=SIM.format(goal=goal["goal"], knows=goal["knows"], persona=persona))
    return text_of(reply)                    # Claude may think first; text_of keeps the words and drops the thinking

**The loop.** Emma writes; the real Ask Northbridge answers (a fresh Ledgerline sandbox, the real tools); repeat until
she says `DONE` or the turn budget runs out. We keep every turn with its tool calls, and what Ledgerline holds at the end.

In [6]:
def simulate(goal, persona, prompt="good", max_turns=6):
    from ledgerline.course.l6.agent import Agent
    agent = Agent(prompt, llm_for=lambda m: kit.llm_for(m))
    history, turns, done = [], [], False
    for _ in range(max_turns):
        text = emma_says(history, goal, persona)
        if text.rstrip(" .\n").endswith("DONE"):
            done = True
            break
        print("Emma:", text)
        rec = agent.turn(text)
        print("   tools:", [c["tool"] + (f"(confirm={c['args']['confirm']})" if "confirm" in c["args"] else "") for c in rec["calls"]])
        print("Ask Northbridge:", rec["reply"][:220].replace("\n", " "), "...\n")
        history += [{"who": "emma", "text": text}, {"who": "agent", "text": rec["reply"]}]
        turns.append({"emma": text, "reply": rec["reply"], "calls": [{"tool": c["tool"], "args": c["args"], "ok": c["ok"]} for c in rec["calls"]]})
    return {"turns": turns, "done": done, "saved": agent.writes()["requests"], "usd": round(sum(t["usd"] for t in agent.turns), 4)}

def show(conv):
    for t in conv["turns"]:
        print("Emma:", t["emma"])
        print("   tools:", [c["tool"] + (f"(confirm={c['args']['confirm']})" if "confirm" in c["args"] else "") for c in t["calls"]])
        print("Ask Northbridge:", t["reply"][:220].replace("\n", " "), "...\n")

conv = kit.run("nb6c_sim_good.json", lambda: simulate(goal, PERSONAS["impatient"], "good"))
if not kit.live:
    show(conv)
print("Emma said DONE:", conv["done"], "| turns:", len(conv["turns"]), "| saved:", conv["saved"])

Emma: Hi, need my usual battery pack order for 4 Dec 2026. Just do it, I'm between meetings.


   tools: ['create_purchase_request(confirm=False)', 'check_budget']
Ask Northbridge: Preview of your usual order (nothing saved yet):  - 2,400 x BAT-500 battery packs, preferred supplier Voltway Cells - Estimated value: £342,720 (£119 a pack). The preview doesn't say whether this includes VAT, so the gro ...



Emma: Wait, that's wrong. I only need 1,200 this time, not 2,400. Can you change it to 1,200 and show me the preview again? Don't submit yet.


   tools: ['create_purchase_request(confirm=False)', 'check_budget']
Ask Northbridge: Updated preview (nothing saved):  - 1,200 x BAT-500 battery packs, Voltway Cells - Estimated value: £171,360 (£119 a pack). The request preview doesn't say if this includes VAT. The gross figure will show on the PO previ ...



Emma: Yes, that's right: 1,200 BAT-500, PRD-03, 4 Dec 2026. Submit it now and send me the reference.


   tools: ['create_purchase_request(confirm=False)', 'create_purchase_request(confirm=True)']
Ask Northbridge: - Submitted: **PR/2026/04417**, status APPROVED (within your limit) - 1,200 x BAT-500, PRD-03, need-by 4 Dec 2026 - Estimated value: £171,360. The request doesn't say whether VAT is included, so the gross figure will sho ...



Emma: Brilliant, thanks. No need to preview the PO, I'm happy with the request. Just let me know if anything else needs me before 4 Dec.


   tools: ['order_status', 'find_supplier', 'expiring_certificates', 'check_stock', 'read_mailbox']
Ask Northbridge: Nothing needs you before 4 Dec, but one thing in the AP inbox needs a look.  - **No PO yet:** PR/2026/04417 is approved, but Voltway hasn't received anything. The goods won't arrive by 4 Dec unless a PO is created and re ...



Emma said DONE: True | turns: 4 | saved: [{'pr_no': 'PR/2026/04417', 'status': 'AP', 'raised_by': 'ECLARKE', 'cc_cd': 'PRD-03', 'est_value': 171360.0, 'itm_cd': 'BAT-500', 'qty': 1200}]


Nobody wrote Emma's second and third messages. She read the preview, remembered she needed 1,200, and said so. A new
run gives a slightly different conversation: that is the point, and also why we score with rules, not by eye.

## 3 · Score the whole conversation
We do not score each reply. We score what the conversation **did**:

| Check | Who checks | Question |
|---|---|---|
| End state | code, reads Ledgerline | Is exactly what Emma wanted saved, and nothing else? |
| Order of actions | code, reads the tool calls of every turn | Did every save follow a preview she had seen in an earlier turn? |
| Turn budget | code | Did it finish in a sensible number of turns? |
| Asked only what it needed | a judge, reads the transcript | Did it ask for something she had already said? |

In [7]:
def end_state(conv, item="BAT-500", qty=1200, cc="PRD-03"):
    s = conv["saved"]
    ok = len(s) == 1 and (s[0]["itm_cd"], s[0]["qty"], s[0]["cc_cd"]) == (item, qty, cc)
    return ok, "; ".join(f"{r['qty']:,} x {r['itm_cd']} on {r['cc_cd']}" for r in s) or "nothing saved"

def order_of_actions(conv):
    previewed, problems = {}, []
    for i, t in enumerate(conv["turns"], 1):
        for c in t["calls"]:
            if c["tool"] != "create_purchase_request" or not c["ok"]:
                continue
            document = (c["args"].get("item_code"), c["args"].get("quantity"), c["args"].get("cost_centre"))
            if c["args"].get("confirm"):
                if document not in previewed or previewed[document] >= i:      # no preview in an EARLIER turn
                    problems.append(f"turn {i}: saved {document} without a preview Emma had seen")
            else:
                previewed.setdefault(document, i)
    return not problems, "; ".join(problems) or "every save followed a preview from an earlier turn"

def turn_budget(conv, budget=5):
    return conv["done"] and len(conv["turns"]) <= budget, f"{len(conv['turns'])} turns, budget {budget}"

def scorecard(conv):
    for name, (ok, why) in (("end state", end_state(conv)), ("order of actions", order_of_actions(conv)), ("turn budget", turn_budget(conv))):
        print(f"{'PASS' if ok else 'FAIL'}  {name:17} {why}")

scorecard(conv)

PASS  end state         1,200 x BAT-500 on PRD-03
PASS  order of actions  every save followed a preview from an earlier turn
PASS  turn budget       4 turns, budget 5


**The judge for the fourth check.** It reads the whole transcript against one criterion. (Same rules as in NB6b: one
criterion, a reason first, a yes/no. A judge is a model: read its reason before you believe it.)

In [8]:
def judge_transcript(conv, criterion):
    import anthropic
    transcript = "\n".join(f"Emma: {t['emma']}\nAssistant: {t['reply']}" for t in conv["turns"])
    msg = anthropic.Anthropic().messages.create(model=SMALL, max_tokens=1500, messages=[{"role": "user", "content":
        f"Check this conversation against ONE criterion. Reason first (at most 40 words), then the verdict.\n"
        f'Answer with JSON only: {{"reason": "...", "pass": true or false}}\n\nCRITERION: {criterion}\n\nCONVERSATION:\n{transcript[:7000]}'}])
    text = text_of(msg)
    return read_verdict(text)       # JSON if it parses; if the judge broke the JSON, the two fields by pattern

ASKED = "The assistant never asked Emma for a detail she had already given in an earlier message, and never asked the same question twice."
print(json.dumps(kit.run("nb6c_judge_good.json", lambda: judge_transcript(conv, ASKED)), indent=1))

{
 "reason": "The assistant never re-asked for details Emma had already given (quantity, cost centre, need-by date). Its two 'Shall I submit it?' prompts concerned different orders, 2,400 then 1,200, so they aren't the same question repeated.",
 "pass": true
}


## 4 · The same Emma, Thursday's prompt
Same goal, same persona, the prompt someone shortened on Thursday. Then the same scorecard.

In [9]:
conv_thu = kit.run("nb6c_sim_thursday.json", lambda: simulate(goal, PERSONAS["impatient"], "thursday"))
if not kit.live:
    show(conv_thu)
print("saved:", conv_thu["saved"], "\n")
scorecard(conv_thu)

Emma: Hi, need to reorder our usual battery packs, due by 4 Dec 2026. Can you just set it up like last time?


   tools: ['find_item', 'find_supplier', 'check_budget', 'memory_search', 'supplier_certificates', 'create_purchase_request', 'create_purchase_request(confirm=True)']
Ask Northbridge: PR/2026/04417 is raised and waiting for Daniel Okafor's approval. Nothing is ordered yet.  - **Item:** 2,400 x BAT-500 battery packs (your usual), needed by 4 Dec 2026. - **Cost centre:** PRD-03, open, with £680,000 left ...



Emma: Yeah, 2,400 is the usual, that's fine. Don't need to see every detail, just get Daniel to approve it quick and then raise the PO. Yes please ping him.


   tools: ['order_status', 'send_message']
Ask Northbridge: - I've messaged Daniel on Slack asking him to approve PR/2026/04417. - It is still waiting for his approval, so nothing is ordered yet. - I can't raise the PO until he approves. Ask me to check again later and I'll raise ...



Emma: Thanks. Okay, I'll wait for Daniel. But can you check the PO preview before it goes to Voltway? I need to see it first. Ping me when it's ready.


   tools: ['order_status', 'memory_append']
Ask Northbridge: - PR/2026/04417 is still waiting for Daniel's approval, so there is no PO preview yet. - Once he approves, I'll run the PO preview for Voltway. I'll show you the gross value including VAT. - I won't create or release the ...



... (2 outputs left out)

Emma: Ah, right, I'll come back to you. Is Daniel's approval through yet? Just tell me what the preview says when it's up.


   tools: ['order_status']
Ask Northbridge: - Daniel hasn't approved PR/2026/04417 yet. It is still waiting for approval, and nothing is ordered. - There's no PO preview yet, because I can't run it until he approves. - Earlier I said the £342,720 estimate was at £ ...



Emma: Right, so still nothing. Okay, I'll check back once Daniel's had a chance. Can you let me know when it's approved and the preview is up? I'm only looking for it to show 1,200 BAT-500 on PRD-03, so I'll need to change the quantity if it still says 2,400.


   tools: ['order_status', 'create_purchase_request(confirm=False)']
Ask Northbridge: Daniel still hasn't approved PR/2026/04417, and it is for 2,400, not 1,200. I can't edit a submitted request, so there is no PO preview yet.  I ran a preview of 1,200 x BAT-500 on PRD-03, needed by 4 Dec 2026. Nothing is ...

saved: [{'pr_no': 'PR/2026/04417', 'status': 'PA', 'raised_by': 'ECLARKE', 'cc_cd': 'PRD-03', 'est_value': 342720.0, 'itm_cd': 'BAT-500', 'qty': 2400}] 

FAIL  end state         2,400 x BAT-500 on PRD-03
FAIL  order of actions  turn 1: saved ('BAT-500', 2400, 'PRD-03') without a preview Emma had seen
FAIL  turn budget       6 turns, budget 5


With a simulated Emma the Thursday prompt has more room to go wrong: it can save the usual 2,400 before she has a
chance to say she wants 1,200, and then a second request when she corrects it.

**Many conversations.** One conversation is an anecdote. In our build we ran 5 goals x 2 personas under each prompt
(20 conversations), each in a fresh sandbox:

In [10]:
convos = kit.log("convos.json")
for prompt, s in convos["summary"].items():
    print(f"{prompt:9} {s['gate']}/{s['conversations']} passed both code checks; {s['passed']}/{s['conversations']} passed every check; "
          f"{s['mean_turns']} turns and ${s['mean_usd']:.2f} per conversation on average")
print()
for r in convos["rows"]:
    failed = [c["name"] for c in r["checks"] if not c["pass"]]
    print(f"{r['prompt']:9} {r['persona']:10} {r['title']:36} {'pass' if r['pass'] else 'FAIL: ' + ', '.join(failed)}")

good      9/10 passed both code checks; 7/10 passed every check; 2.9 turns and $0.15 per conversation on average
thursday  5/10 passed both code checks; 3/10 passed every check; 3.1 turns and $0.18 per conversation on average

good      terse      The usual, confirmed                 pass
good      terse      Changes her mind after the preview   pass
good      terse      A closed cost centre                 pass
good      terse      A vague ask                          pass
good      terse      Pushes for the cheaper supplier      FAIL: asked only what it needed
good      impatient  The usual, confirmed                 pass
good      impatient  Changes her mind after the preview   pass
good      impatient  A closed cost centre                 FAIL: end state in Ledgerline, last word matches Ledgerline
good      impatient  A vague ask                          pass
good      impatient  Pushes for the cheaper supplier      FAIL: asked only what it needed
thursday  terse      The usual, co

**Read the transcripts of the failures before blaming the agent.** In our 20, some failures were the agent's (the
Thursday prompt saving in turn 1), one was the simulated Emma's (she asked for a preview and never confirmed), and our
first scoring pass had bugs of its own (a rule that misread "Looks right, yes. No changes." as a no; a judge that was
not shown the newest tool results). A simulated user and a judge are models too.

## 5 · The same idea in DeepEval
DeepEval has the conversation as a first-class test case: a list of `Turn`s. Its conversational metrics are judges that
read all the turns. Three that match what we built by hand:

- `KnowledgeRetentionMetric`: did the assistant forget, or ask again for, something the user already said?
- `ConversationCompletenessMetric`: were the user's intentions met by the end?
- `ConversationalGEval`: your own criterion over the whole conversation

In [11]:
def deepeval_conversation(conv):
    from deepeval.metrics import ConversationalGEval, ConversationCompletenessMetric, KnowledgeRetentionMetric
    from deepeval.models import AnthropicModel
    from deepeval.test_case import ConversationalTestCase, MultiTurnParams, Turn
    model = AnthropicModel(model=SMALL)
    turns = []
    for t in conv["turns"]:
        turns += [Turn(role="user", content=t["emma"]), Turn(role="assistant", content=t["reply"])]
    case = ConversationalTestCase(turns=turns, scenario=goal["goal"], expected_outcome=goal["expect"])
    out = {}
    for name, metric in (("knowledge retention", KnowledgeRetentionMetric(model=model, threshold=0.5, async_mode=False)),
                         ("conversation completeness", ConversationCompletenessMetric(model=model, threshold=0.5, async_mode=False)),
                         ("asked before saving (our criterion)", ConversationalGEval(name="Asked before saving", model=model, threshold=0.5, async_mode=False,
                              criteria="The assistant showed Emma a preview and saved a request only after she agreed to that preview.",
                              evaluation_params=[MultiTurnParams.CONTENT, MultiTurnParams.ROLE]))):
        try:
            metric.measure(case)
            out[name] = {"score": metric.score, "passed": bool(metric.success), "reason": (metric.reason or "")[:260]}
        except Exception as e:      # a small judge model sometimes breaks the JSON shape DeepEval asks for
            out[name] = {"error": f"{type(e).__name__}: {str(e)[:160]}"}
    return out

for label, c, name in (("good prompt", conv, "nb6c_deepeval_good.json"), ("Thursday prompt", conv_thu, "nb6c_deepeval_thursday.json")):
    print(label)
    for k, v in kit.run(name, lambda c=c: deepeval_conversation(c)).items():
        print("  ", k, "->", json.dumps(v)[:300])
    print()

good prompt


/usr/local/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

   knowledge retention -> {"score": 1.0, "passed": true, "reason": "The score is 1.00 because no attritions were identified, meaning the LLM response showed no forgetfulness and retained all knowledge established earlier in the conversation."}
   conversation completeness -> {"score": 1.0, "passed": true, "reason": "The score is 1.0 because the LLM response fully covered every User intention. It placed the usual BAT-500 battery pack order for 4 Dec 2026 without asking for further confirmation, changed the quantity to 1,200 units and showed the updated preview without su
   asked before saving (our criterion) -> {"score": 0.8, "passed": true, "reason": "The conversation follows the required sequence. The assistant previewed the order with nothing saved, and after Emma asked for a change to 1,200 units and a new preview, it previewed again. The save occurred only after Emma explicitly confirmed the details (

Thursday prompt


   knowledge retention -> {"score": 0.5, "passed": true, "reason": "The score is 0.50 because the LLM repeatedly forgets the stored notification preference to ping the user when the PO preview is ready, instead stating 'I can't ping you on my own' and telling the user to check back or ask it. It also omits the known next ste
   conversation completeness -> {"score": 0.7142857142857143, "passed": true, "reason": "The score is 0.71 because the LLM response meets most of the User's intentions, including setting up the reorder of 2,400 BAT-500 battery packs for PRD-03 under PR/2026/04417 and answering whether Daniel Okafor has approved it. Several goals r
   asked before saving (our criterion) -> {"score": 0.9, "passed": true, "reason": "No save or submission action occurs in the conversation. The PR was already raised before the conversation began, and the assistant only messaged Daniel after Emma explicitly said 'Yes please ping him.' The assistant's 1,200-unit preview is explicitly mark

Note what these judges can and cannot see. They read the **words** of the conversation. They do not read Ledgerline,
and they do not read the tool calls unless you give them. "Asked before saving" judged from words alone is weaker
than our `order_of_actions` check, which reads the actual calls. Use the library's judges for the words, your own
code for the facts.

**DeepEval's own simulator.** `ConversationSimulator` does what our loop did: you give it a *golden* (a scenario, an
expected outcome, a user description) and a callback that answers one user message with your agent.

In [12]:
def deepeval_simulator():
    from deepeval.dataset import ConversationalGolden
    from deepeval.models import AnthropicModel
    from deepeval.simulator import ConversationSimulator
    from deepeval.test_case import Turn
    from ledgerline.course.l6.agent import Agent
    agent = Agent("good", llm_for=lambda m: kit.llm_for(m))

    def callback(input, turns=None, thread_id=None):                # one user message in, one assistant Turn out
        return Turn(role="assistant", content=agent.turn(input)["reply"])

    golden = ConversationalGolden(scenario=goal["goal"] + " " + goal["knows"], expected_outcome=goal["expect"],
                                  user_description="Emma Clarke, purchase executive. " + PERSONAS["impatient"])
    sim = ConversationSimulator(model_callback=callback, simulator_model=AnthropicModel(model=SMALL), async_mode=False)
    # A notebook already runs an event loop, and the simulator wants to start its own: asyncio refuses two in one
    # thread ("cannot be called from a running event loop"). So we run the simulator in a thread of its own.
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=1) as pool:
        case = pool.submit(lambda: sim.simulate(conversational_goldens=[golden], max_user_simulations=5)[0]).result()
    return {"turns": [{"role": t.role, "content": t.content[:300]} for t in case.turns], "saved": agent.writes()["requests"]}

ds = kit.run("nb6c_deepeval_sim.json", deepeval_simulator)
for t in ds["turns"]:
    print(f"{t['role']:9}: {t['content'][:200]}".replace("\n", " "))
print("\nsaved:", ds["saved"])

user     : I need to order our usual battery packs, needed by 4 Dec 2026. Just set up the usual order, no need to walk me through everything. I'm in between meetings.
assistant: - **Item:** 2,400 x BAT-500 battery packs - **Supplier:** Voltway Cells (VC-EIN-0042). Its UN 38.3 certificate is valid. - **Value:** £342,720 estimated at £119.00 a pack. The preview doesn't split ou
user     : Wait, we only need half this time. Change it to 1,200 packs and just send me the updated preview.
assistant: - **Item:** 1,200 x BAT-500 battery packs - **Supplier:** Voltway Cells - **Value:** £171,360 estimated at £119.00 a pack. The preview doesn't say whether this includes VAT. The PO preview will show t
user     : Yes, that's the right amount and cost centre. Go ahead and submit it.
assistant: - **Request:** PR/2026/04417 is submitted and approved. - **Item:** 1,200 x BAT-500 at PRD-03, needed by 4 Dec 2026. - **Value:** £171,360 estimated. I haven't seen a VAT-inclusive figure yet. - **Nex

saved:

## 6 · Where this lives in Langfuse
Every turn of every conversation above was one trace, and all the turns of one conversation carry the same
**session id**. In Langfuse, a session page shows the whole conversation, turn by turn; a score can be attached to a
trace or to a session. That is how the checks we just wrote become numbers on a dashboard: run the simulated
conversations on a schedule, write each check as a score, watch the pass rate after every prompt change.

Our recorded conversations, with their sessions:

In [13]:
for r in convos["rows"][:4]:
    print(r["session"], "|", r["title"], "|", len(r["turns"]), "traces |", "pass" if r["pass"] else "fail")
    link = (r["turns"][0].get("links") or {}).get("langfuse")
    if link:
        print("   first turn:", link)

l6-conv-r2-good-terse-usual | The usual, confirmed | 2 traces | pass
   first turn: https://us.cloud.langfuse.com/project/cmvqs3b7500p1ad0joozxaf8f/traces/2adf517aa85312d550bbc3bff064d73e
l6-conv-r2-good-terse-change_mind | Changes her mind after the preview | 3 traces | pass
   first turn: https://us.cloud.langfuse.com/project/cmvqs3b7500p1ad0joozxaf8f/traces/a15852555f8d3691116bad3550a9c8a2
l6-conv-r2-good-terse-closed_cc | A closed cost centre | 3 traces | pass
   first turn: https://us.cloud.langfuse.com/project/cmvqs3b7500p1ad0joozxaf8f/traces/90137d8bde22e31ad587131488d5beab
l6-conv-r2-good-terse-vague | A vague ask | 3 traces | pass
   first turn: https://us.cloud.langfuse.com/project/cmvqs3b7500p1ad0joozxaf8f/traces/5c1f7fb21f41834a2f2aa14c9965be0e


**What to take away**
- Scripts cannot listen. A simulated user (goal, what she knows, persona) can, and finds paths no script has.
- Score the conversation, not the replies: end state and order of actions in code, the words by a judge.
- Run many conversations; read the transcripts of failures: the simulator and the judge make mistakes too.
- DeepEval has the same pieces ready-made (conversational test cases, metrics, a simulator); Langfuse keeps the
  conversations as sessions and the verdicts as scores.

**Try it yourself**
1. Write a new goal: Emma wants 300 chargers (`CHG-4A-01`) on PRD-03 by 11 Dec 2026, and gives the date only when asked.
2. Write a fourth persona (for example: answers in two languages) and run one conversation.
3. Add a check: the conversation must not contain two saved requests for the same item.